# CreditWise AI — Data Cleaning and Preprocessing

This notebook creates a documented, repeatable preprocessing workflow from the original CSV. It removes exact duplicate records, excludes only ages confirmed to be impossible in this dataset, and treats two impossible employment-length entries as missing. Other extreme income and loan amounts are retained. The original CSV is read-only throughout.

The cleaned CSV and train/test CSVs remain **untransformed** and retain missing feature values. Imputation, scaling, and one-hot encoding are configured in a scikit-learn `ColumnTransformer`; it is fit on the training features only. No estimator is trained. The fitted preprocessor and a JSON configuration (including source row IDs for the split) are saved to `data/processed/`.

## 1. Verify the EDA findings against the original CSV

The assertions below make the EDA findings reproducible checks. The unusual records are displayed before any decision is applied.

In [ ]:
from pathlib import Path
import hashlib
import json
import sys

import joblib
import numpy as np
import pandas as pd
import sklearn
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

candidate_roots = [Path.cwd(), *Path.cwd().parents]
project_root = next(
    (root for root in candidate_roots if (root / "data/raw/credit_risk_dataset.csv").is_file()),
    None,
)
if project_root is None:
    raise FileNotFoundError("Could not locate data/raw/credit_risk_dataset.csv from this notebook's working directory.")

raw_path = project_root / "data/raw/credit_risk_dataset.csv"
processed_dir = project_root / "data/processed"
processed_dir.mkdir(parents=True, exist_ok=True)
original_sha256 = hashlib.sha256(raw_path.read_bytes()).hexdigest()
df_original = pd.read_csv(raw_path)
assert ".venv" in str(Path(sys.executable)).lower(), f"Expected project .venv kernel, got {sys.executable}"
assert df_original.shape == (32_581, 12), f"Unexpected original shape: {df_original.shape}"
assert int(df_original.duplicated().sum()) == 165
assert int(df_original["loan_int_rate"].isna().sum()) == 3_116
assert int(df_original["person_emp_length"].isna().sum()) == 895
assert int((df_original["person_age"] > 100).sum()) == 5
assert int((df_original["person_emp_length"] > df_original["person_age"]).sum()) == 2
assert "loan_status" in df_original.columns

print(f"Project root: {project_root}")
print(f"Python kernel: {sys.executable} ({sys.version.split()[0]})")
print(f"Original CSV: {df_original.shape[0]:,} rows × {df_original.shape[1]} columns")
print(f"Target variable: loan_status; distribution: {df_original['loan_status'].value_counts().sort_index().to_dict()}")
print(f"Verified duplicates: {df_original.duplicated().sum():,}")
print("Verified missing values:", df_original.isna().sum()[df_original.isna().sum() > 0].to_dict())
print(f"Verified age > 100: {(df_original['person_age'] > 100).sum():,}; employment length > age: {(df_original['person_emp_length'] > df_original['person_age']).sum():,}")

In [ ]:
age_anomalies = df_original.loc[df_original["person_age"] > 100].copy()
employment_anomalies = df_original.loc[
    df_original["person_emp_length"] > df_original["person_age"]
].copy()

review_columns = [
    "person_age", "person_income", "person_emp_length", "loan_intent",
    "loan_grade", "loan_amnt", "loan_status",
]
print("Age records requiring review (source CSV row IDs are the displayed index):")
display(age_anomalies[review_columns])
print("Employment-length records requiring review:")
display(employment_anomalies[review_columns])

print("Evidence-based decisions:")
print("- The five ages are 123 or 144 years. Exclude these records from the analysis dataset as implausible borrower ages.")
print("- The two employment lengths are each 123 years for borrowers aged 21 and 22. Set only person_emp_length to missing; retain the borrowers and impute from the training set later.")
print("- Remove exact duplicate rows, retaining the first occurrence.")
print("- Preserve other values, including extreme incomes and loan amounts; no arbitrary winsorization or range clipping is applied.")
print("- Existing feature missing values remain missing in saved cleaned/split CSVs; imputation is performed only by the training-fitted preprocessing pipeline.")

## 2. Clean records and save an untransformed dataset

Cleaning is limited to the documented record-level decisions. The output retains the original 12 columns and any valid missing feature values. `loan_status` is neither imputed nor changed.

In [ ]:
cleaned_df = df_original.drop_duplicates(keep="first").copy()
duplicate_rows_removed = len(df_original) - len(cleaned_df)

impossible_age_mask = cleaned_df["person_age"] > 100
age_rows_excluded = int(impossible_age_mask.sum())
cleaned_df = cleaned_df.loc[~impossible_age_mask].copy()

impossible_employment_mask = cleaned_df["person_emp_length"] > cleaned_df["person_age"]
employment_values_set_missing = int(impossible_employment_mask.sum())
cleaned_df.loc[impossible_employment_mask, "person_emp_length"] = np.nan

assert duplicate_rows_removed == 165
assert age_rows_excluded == 5
assert employment_values_set_missing == 2
assert cleaned_df.shape == (32_411, 12)
assert not cleaned_df.duplicated().any()
assert not (cleaned_df["person_age"] > 100).any()
assert not (cleaned_df["person_emp_length"] > cleaned_df["person_age"]).any()
assert cleaned_df["loan_status"].notna().all()
assert int(cleaned_df["loan_int_rate"].isna().sum()) == 3_116
assert int(cleaned_df["person_emp_length"].isna().sum()) == 897

cleaned_path = processed_dir / "credit_risk_cleaned.csv"
cleaned_df.to_csv(cleaned_path, index=False)

record_counts = pd.DataFrame({
    "stage": ["Original", "After exact duplicate removal", "After age review/exclusion"],
    "records": [len(df_original), len(df_original) - duplicate_rows_removed, len(cleaned_df)],
})
display(record_counts)
print(f"Removed exact duplicates: {duplicate_rows_removed:,} (kept first copy).")
print(f"Excluded impossible-age records: {age_rows_excluded:,} (age > 100).")
print(f"Set impossible employment-length values to missing: {employment_values_set_missing:,}.")
print(f"Cleaned untransformed data saved to: {cleaned_path}")
print("Retained extreme income/loan values:", {
    "maximum_person_income": int(cleaned_df["person_income"].max()),
    "maximum_loan_amount": int(cleaned_df["loan_amnt"].max()),
})

## 3. Separate features and target; create a stratified split

`loan_status` is kept as `y` and never included in `X`. The 80:20 split is stratified so each partition preserves the observed class proportions. Split files are untransformed. Their source row IDs are also saved in the configuration file for exact provenance.

In [ ]:
target_column = "loan_status"
feature_columns = [column for column in cleaned_df.columns if column != target_column]
X = cleaned_df[feature_columns].copy()
y = cleaned_df[target_column].copy()
assert target_column not in X.columns
assert y.notna().all(), "Target values must be present; target imputation is not allowed."

numerical_features = X.select_dtypes(include="number").columns.tolist()
categorical_features = X.select_dtypes(exclude="number").columns.tolist()
print(f"Numerical features ({len(numerical_features)}): {numerical_features}")
print(f"Categorical features ({len(categorical_features)}): {categorical_features}")
print(f"Target: {target_column}; excluded from features: {target_column not in feature_columns}")
print("Leakage/availability review flags:")
print("- loan_grade is retained for comparison, but may encode the lender's original underwriting assessment; compare models with and without it.")
print("- loan_int_rate and loan_percent_income may also reflect underwriting decisions or information availability; verify prediction-time availability and intended use.")
print("- cb_person_default_on_file is historical credit information; confirm it is available at the intended decision point and assess for proxy/fairness concerns.")

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y,
)

train_df = X_train.copy()
train_df[target_column] = y_train
test_df = X_test.copy()
test_df[target_column] = y_test
train_path = processed_dir / "credit_risk_train.csv"
test_path = processed_dir / "credit_risk_test.csv"
train_df.to_csv(train_path, index=False)
test_df.to_csv(test_path, index=False)

def target_distribution(labels):
    counts = labels.value_counts().reindex([0, 1], fill_value=0)
    return pd.DataFrame({"count": counts, "percent": counts.div(len(labels)).mul(100)})

print(f"Train: {len(X_train):,}; test: {len(X_test):,}; combined: {len(X_train) + len(X_test):,}")
print("Training target distribution:")
display(target_distribution(y_train).style.format({"count": "{:,.0f}", "percent": "{:.2f}%"}))
print("Testing target distribution:")
display(target_distribution(y_test).style.format({"count": "{:,.0f}", "percent": "{:.2f}%"}))
print(f"Untransformed stratified splits saved to {train_path} and {test_path}.")

## 4. Define and fit preprocessing on training features only

Numeric columns use median imputation followed by standard scaling. Categorical columns use most-frequent imputation followed by one-hot encoding; `handle_unknown="ignore"` allows new categories during later transforms without an error. The `ColumnTransformer` is fit only on `X_train`; transforming `X_test` uses those learned training statistics without refitting. These transformed matrices are only checked in memory and are not saved.

In [ ]:
numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])
categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer(transformers=[
    ("numeric", numeric_pipeline, numerical_features),
    ("categorical", categorical_pipeline, categorical_features),
], remainder="drop")

# Fit only on training data. Test data is transformed, never used to fit statistics.
X_train_transformed = preprocessor.fit_transform(X_train)
X_test_transformed = preprocessor.transform(X_test)
assert X_train_transformed.shape[0] == len(X_train)
assert X_test_transformed.shape[0] == len(X_test)
assert X_train_transformed.shape[1] == X_test_transformed.shape[1]

preprocessor_path = processed_dir / "credit_risk_preprocessor.joblib"
joblib.dump(preprocessor, preprocessor_path)

fitted_numeric_imputer = preprocessor.named_transformers_["numeric"].named_steps["imputer"]
fitted_scaler = preprocessor.named_transformers_["numeric"].named_steps["scaler"]
fitted_categorical_imputer = preprocessor.named_transformers_["categorical"].named_steps["imputer"]
fitted_encoder = preprocessor.named_transformers_["categorical"].named_steps["onehot"]
print(f"Transformed training matrix shape (in memory only): {X_train_transformed.shape}")
print(f"Transformed test matrix shape (in memory only): {X_test_transformed.shape}")
print("Learned training-only numeric medians:", dict(zip(numerical_features, fitted_numeric_imputer.statistics_.tolist())))
print("Learned training-only numeric scaler means:", dict(zip(numerical_features, fitted_scaler.mean_.tolist())))
print("Learned one-hot categories:", {
    column: categories.tolist()
    for column, categories in zip(categorical_features, fitted_encoder.categories_)
})
print(f"Fitted preprocessing transformer saved to: {preprocessor_path}")
print("No estimator/model was trained; transformed arrays were not written to disk.")

## 5. Save reproducibility configuration and summarize decisions

The JSON configuration records the input fingerprint, cleaning policy, feature lists, split settings and exact source-row membership, preprocessing steps, and fitted training-only imputation/scaling statistics. The joblib artifact contains the fitted transformer. Together with the untransformed train/test CSVs, these make the preparation reproducible without deriving any statistics from the test set.

In [ ]:
config = {
    "source_file": str(raw_path.relative_to(project_root)),
    "source_sha256": original_sha256,
    "original_shape": {"rows": int(len(df_original)), "columns": int(df_original.shape[1])},
    "cleaning": {
        "exact_duplicate_rows_removed": duplicate_rows_removed,
        "duplicate_policy": "drop_duplicates(keep='first')",
        "age_threshold": 100,
        "age_rows_excluded": age_rows_excluded,
        "employment_length_greater_than_age_values_set_missing": employment_values_set_missing,
        "other_income_and_loan_amount_extremes": "retained; no clipping or winsorization",
        "other_missing_values": "retained in untransformed outputs; imputed in fitted training-only transformer",
        "target": "loan_status is never imputed and is not included in X",
    },
    "cleaned_file": str(cleaned_path.relative_to(project_root)),
    "cleaned_shape": {"rows": int(len(cleaned_df)), "columns": int(cleaned_df.shape[1])},
    "target_column": target_column,
    "feature_columns": feature_columns,
    "numerical_features": numerical_features,
    "categorical_features": categorical_features,
    "split": {
        "test_size": 0.20,
        "train_size": 0.80,
        "random_state": 42,
        "stratify_column": target_column,
        "train_rows": int(len(X_train)),
        "test_rows": int(len(X_test)),
        "train_target_counts": {str(key): int(value) for key, value in y_train.value_counts().sort_index().items()},
        "test_target_counts": {str(key): int(value) for key, value in y_test.value_counts().sort_index().items()},
        "train_source_row_ids": [int(value) for value in X_train.index],
        "test_source_row_ids": [int(value) for value in X_test.index],
        "train_file": str(train_path.relative_to(project_root)),
        "test_file": str(test_path.relative_to(project_root)),
    },
    "preprocessing": {
        "numeric": {"imputation": "median", "scaling": "StandardScaler"},
        "categorical": {"imputation": "most_frequent", "encoding": "OneHotEncoder", "handle_unknown": "ignore"},
        "fit_data": "X_train only",
        "transformed_data_saved": False,
        "fitted_transformer_file": str(preprocessor_path.relative_to(project_root)),
        "training_numeric_imputer_statistics": dict(zip(numerical_features, [float(value) for value in fitted_numeric_imputer.statistics_])),
        "training_numeric_scaler_mean": dict(zip(numerical_features, [float(value) for value in fitted_scaler.mean_])),
        "training_numeric_scaler_scale": dict(zip(numerical_features, [float(value) for value in fitted_scaler.scale_])),
        "training_categorical_imputer_statistics": dict(zip(categorical_features, [str(value) for value in fitted_categorical_imputer.statistics_])),
        "onehot_categories": {column: [str(value) for value in categories] for column, categories in zip(categorical_features, fitted_encoder.categories_)},
    },
    "versions": {"python": sys.version.split()[0], "pandas": pd.__version__, "numpy": np.__version__, "scikit_learn": sklearn.__version__},
    "leakage_review": {
        "target_excluded_from_features": target_column not in feature_columns,
        "loan_grade": "retained for later with/without comparison; may represent lender's original underwriting assessment",
        "other_features_to_review": ["loan_int_rate", "loan_percent_income", "cb_person_default_on_file"],
    },
}
config_path = processed_dir / "credit_risk_preprocessing_config.json"
config_path.write_text(json.dumps(config, indent=2), encoding="utf-8")

current_sha256 = hashlib.sha256(raw_path.read_bytes()).hexdigest()
assert current_sha256 == original_sha256, "Original CSV changed during this workflow."
assert len(X_train) + len(X_test) == len(cleaned_df)
assert set(X_train.index).isdisjoint(set(X_test.index))
assert target_column not in X_train.columns and target_column not in X_test.columns
assert cleaned_path.is_file() and train_path.is_file() and test_path.is_file()
assert preprocessor_path.is_file() and config_path.is_file()

print("Final record-count summary:")
print(f"- Original: {len(df_original):,}")
print(f"- Cleaned: {len(cleaned_df):,} (165 duplicates removed; 5 impossible-age records excluded)")
print(f"- Training: {len(X_train):,}; testing: {len(X_test):,} (80:20 stratified split, random_state=42)")
print(f"- Employment anomalies retained as rows with missing person_emp_length: {employment_values_set_missing:,}")
print("- Leakage prevention: target excluded from X; split before fitting; preprocessing fit only on X_train; test only transformed.")
print(f"- Original CSV unchanged (SHA-256 verified): {current_sha256}")
print("Saved artifacts:")
for path in [cleaned_path, train_path, test_path, preprocessor_path, config_path]:
    print(f"  {path.relative_to(project_root)} ({path.stat().st_size:,} bytes)")